# Netflix Content Analysis: A Stepwise Graph Exploration <a target="_blank" href="https://colab.research.google.com/github/yWorks/yfiles-jupyter-graphs/blob/main/examples/showcases/netflix_movies.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this notebook, we explore the [Netflix Shows dataset](https://www.kaggle.com/datasets/shivamb/netflix-shows) through a series of focused graph visualizations.

Large graphs can quickly become unreadable. By breaking the data down into specific "aspects" (Genres, Geography, Talent, etc.), we can derive clearer insights and patterns.

We use the **yfiles_jupyter_graphs** library to create interactive visualizations.

### Before using the widget, make sure to install the required packages

Ensure you have the necessary packages installed by running the following command:
- ```%pip install yfiles_jupyter_graphs kagglehub --quiet```

In [ ]:
%pip install yfiles_jupyter_graphs kagglehub pandas --quiet

You can also open this notebook in Google Colab when Google Colab's custom widget manager is enabled:

In [ ]:
try:
  import google.colab
  from google.colab import output
  output.enable_custom_widget_manager()
except:
  pass

<a target="_blank" href="https://colab.research.google.com/github/yWorks/yfiles-jupyter-graphs/blob/main/examples/showcases/netflix_movies.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub
import pandas as pd
import numpy as np
from yfiles_jupyter_graphs import GraphWidget, Node, Edge, Layout
from yfiles_jupyter_graphs import NodeStyle, NodeShape
from IPython.display import display

In [ ]:
# Download and load data
path = kagglehub.dataset_download("shivamb/netflix-shows")
df = pd.read_csv(path + "/netflix_titles.csv")

# Pre-processing: Filter for 2021, drop NaNs, and select first 100
subset = df[df['release_year'].astype(str).str.strip() == '2021'].dropna()
print(f"Analyzing the top {len(subset)} shows from 2021.")
subset.head()

## Graph Building Logic

To maintain consistency across our visualizations, we define a helper function that builds a graph based on requested attributes.

In [ ]:
def build_netflix_graph(data, include_genres=False, include_countries=False, include_people=False, include_ratings=False, include_years=False, group_by_genre=False):
    nodes = []
    edges = []
    seen_nodes = set()

    def add_node(id_, label, type_, **props):
        if id_ not in seen_nodes:
            p = {"label": label, "type": type_}
            p.update(props)
            nodes.append(Node(id=id_, properties=p))
            seen_nodes.add(id_)

    for _, row in data.iterrows():
        show_id = f"show::{row['show_id']}"
        show_props = {"release_year": row['release_year']}

        if include_genres and pd.notna(row['listed_in']):
            genres = [g.strip() for g in row['listed_in'].split(', ')]
            if group_by_genre and genres:
                # When grouping by genre, only the first genre acts as a parent.
                # To avoid empty genre nodes, we only add the primary genre.
                primary_genre = genres[0]
                show_props["parent_id"] = f"genre::{primary_genre}"
                add_node(f"genre::{primary_genre}", primary_genre, "genre")
            elif not group_by_genre:
                for g in genres:
                    add_node(f"genre::{g}", g, "genre")
                    edges.append(Edge(start=show_id, end=f"genre::{g}", properties={"edge_type": "genre"}))

        if include_countries and pd.notna(row['country']):
            for c in row['country'].split(', '):
                c = c.strip()
                add_node(f"country::{c}", c, "country")
                edges.append(Edge(start=show_id, end=f"country::{c}", properties={"edge_type": "country"}))

        if include_people:
            if pd.notna(row['director']):
                for d in row['director'].split(', '):
                    d = d.strip()
                    add_node(f"director::{d}", d, "director")
                    edges.append(Edge(start=show_id, end=f"director::{d}", properties={"edge_type": "director"}))
            if pd.notna(row['cast']):
                # Top 3 actors to keep it manageable
                for a in [a.strip() for a in row['cast'].split(', ')][:3]:
                    add_node(f"actor::{a}", a, "actor")
                    edges.append(Edge(start=show_id, end=f"actor::{a}", properties={"edge_type": "actor"}))

        if include_ratings and pd.notna(row['rating']):
            r = row['rating'].strip()
            add_node(f"rating::{r}", r, "rating")
            edges.append(Edge(start=show_id, end=f"rating::{r}", properties={"edge_type": "rating"}))

        if include_years:
            y = str(row['release_year'])
            add_node(f"year::{y}", y, "year")
            edges.append(Edge(start=show_id, end=f"year::{y}", properties={"edge_type": "year"}))

        add_node(show_id, row['title'], row['type'], **show_props)

    w = GraphWidget(nodes=nodes, edges=edges, directed=False)

    # Standard Mappings
    w.node_label_mapping = "label"

    def style_mapping(n):
        props = n.get("properties", {})
        t = props.get("type", "unknown")
        if t == "Movie": return NodeStyle(color="#E50914", shape=NodeShape.ROUND_RECTANGLE)
        if t == "TV Show": return NodeStyle(color="#B20710", shape=NodeShape.ROUND_RECTANGLE)
        if t == "genre": return NodeStyle(color="#4CAF50", shape=NodeShape.HEXAGON)
        if t == "country": return NodeStyle(color="#2196F3", shape=NodeShape.ELLIPSE)
        if t == "director": return NodeStyle(color="#9C27B0", shape=NodeShape.PILL)
        if t == "actor": return NodeStyle(color="#FF9800", shape=NodeShape.RECTANGLE)
        if t == "rating": return NodeStyle(color="#607D8B", shape=NodeShape.TRIANGLE)
        if t == "year": return NodeStyle(color="#795548", shape=NodeShape.OCTAGON)
        return NodeStyle(color="#CCCCCC")

    w.node_styles_mapping = style_mapping

    if group_by_genre:
        w.node_parent_mapping = lambda n: n.get("properties", {}).get("parent_id")

    w.edge_color_mapping = lambda e: {
        "genre": "#4CAF50", "country": "#2196F3", "director": "#9C27B0",
        "actor": "#FF9800", "rating": "#607D8B", "year": "#795548"
    }.get(e.get("properties", {}).get("edge_type"), "#AAAAAA")

    return w

## Aspect 1: The Genre Landscape

Which genres are most common in the latest additions? This graph connects shows to their categories. Large hubs among genre nodes indicate popular themes like "International TV Shows" or "Dramas".

In [ ]:
w1 = build_netflix_graph(subset, include_genres=True, group_by_genre=True)
w1.graph_layout = Layout.ORGANIC
display(w1)

## Aspect 2: Global Production Hubs

Netflix is a global platform. By connecting shows to their countries of origin, we can see which nations are the primary content producers and identify international co-productions (shows connected to multiple countries).

In this graph, we use **data-driven scaling**. We calculate how many shows each country in our subset has produced and scale the country nodes accordingly. Larger nodes represent major production hubs.

In [ ]:
w2 = build_netflix_graph(subset, include_countries=True)

# Calculate production volume per country
country_counts = {}
for _, row in subset.iterrows():
    if pd.notna(row['country']):
        for c in row['country'].split(', '):
            c = c.strip()
            country_counts[c] = country_counts.get(c, 0) + 1

w2.node_scale_factor_mapping = lambda n: 1.0 + (country_counts.get(n.get('properties', {}).get('label'), 0) * 0.2) if n.get('properties', {}).get('type') == 'country' else 0.8
w2.graph_layout = Layout.ORGANIC
display(w2)

## Aspect 3: Content Ratings & Audience Segments

How is the latest content rated? This graph shows the distribution across ratings like TV-MA, TV-14, etc. It helps visualize which audience segments Netflix is currently targeting.

In [ ]:
w3 = build_netflix_graph(subset, include_ratings=True)
w3.graph_layout = Layout.CIRCULAR
display(w3)

## Aspect 4: Deep Dive - The "Documentaries" Niche

Finally, let's look at a specific subset: Documentaries. We filter the data to only include shows listed as Documentaries and see their directors and countries in one view.

In [ ]:
doc_subset = subset[subset['listed_in'].str.contains('Documentaries', na=False)]
w4 = build_netflix_graph(doc_subset, include_countries=True, include_people=True)
w4.edge_label_mapping = lambda edge: edge['properties']['edge_type']
w4.graph_layout = Layout.ORGANIC
display(w4)

## Aspect 5: Interactive Master Exploration

To conclude, we present a comprehensive graph containing Genres, Countries, and Ratings. We enable the **sidebar search** and **overview** to help you navigate this more complex view.

In [ ]:
w5 = build_netflix_graph(subset, include_genres=True, include_countries=True, include_ratings=True)
w5.sidebar = {"enabled": True, "start_with": "Search"}
w5.overview = True
w5.graph_layout = Layout.ORGANIC
display(w5)